# Emotions dataset → tiered LTS pipeline (GitHub + Colab)

This notebook uses the repo as the **source of truth** for code and prompts.

**Pipeline overview**

1. Mount Google Drive and clone / pull the GitHub branch  
2. Set experiment paths inside the repository  
3. Load `dair-ai/emotion` from HuggingFace  
4. Build the **6-class** train / val / test CSVs (`TARGET_SLUG = "emotion"`) under `data/processed/`  
5. Write `prompts/few_shot_examples_emotion.json`  
6. Verify Python modules under `src/` (including `tiered_labels.py`)  
7. Reset sampler state files before a fresh run  
8. Train/eval with **`main_cluster_hierarchical.py`** (tier 1: pos/neg; tier 2: neg→sadness/anger/fear, pos→joy/love)

See also `emotions_rec/COLAB.md` for a written Colab guide.

Authoritative code lives in `emotions_rec/src/`, `emotions_rec/prompts/`, and this notebook folder. Generated CSVs go to `data/processed/`; checkpoints and state files are local to the experiment directory.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import sys

REPO_URL = "https://github.com/ShenyaoZhang/DS-GA-3001-Data-Engineering-Project.git"
BRANCH = "emotions-rec"

DRIVE_ROOT = "/content/drive/MyDrive"
REPO_ROOT = os.path.join(DRIVE_ROOT, "DS-GA-3001-Data-Engineering-Project")
EXPERIMENT_ROOT = os.path.join(REPO_ROOT, "emotions_rec")

SRC_DIR = os.path.join(EXPERIMENT_ROOT, "src")
PROMPTS_DIR = os.path.join(EXPERIMENT_ROOT, "prompts")
DATA_DIR = os.path.join(EXPERIMENT_ROOT, "data")
DATA_RAW_DIR = os.path.join(DATA_DIR, "raw")
DATA_PROCESSED_DIR = os.path.join(DATA_DIR, "processed")
OUTPUTS_DIR = os.path.join(EXPERIMENT_ROOT, "outputs")

if not os.path.exists(REPO_ROOT):
    print("Cloning repository into Google Drive...")
    !git clone -b {BRANCH} {REPO_URL} "{REPO_ROOT}"
else:
    print("Repository already exists in Drive. Pulling latest changes...")
    %cd "{REPO_ROOT}"
    !git fetch origin
    !git checkout {BRANCH}
    !git pull

%cd "{EXPERIMENT_ROOT}"

os.makedirs(DATA_RAW_DIR, exist_ok=True)
os.makedirs(DATA_PROCESSED_DIR, exist_ok=True)
os.makedirs(OUTPUTS_DIR, exist_ok=True)

if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

print("Experiment root:", EXPERIMENT_ROOT)
print("Processed data dir:", DATA_PROCESSED_DIR)
print("Prompts dir:", PROMPTS_DIR)
print("Source dir:", SRC_DIR)
print("Files in experiment root:", sorted(os.listdir(EXPERIMENT_ROOT)))


In [ ]:
from getpass import getpass
import os

token = getpass("Enter your HF token: ")
os.environ["HF_TOKEN"] = token
os.environ["HUGGINGFACE_HUB_TOKEN"] = token
print("HF token set.")

## 1) Install dependencies

This notebook installs dependencies from the repository `requirements.txt` so that the environment matches the experiment branch as closely as possible.


In [ ]:
!pip -q install -r "{os.path.join(REPO_ROOT, 'requirements.txt')}"
!pip -q install datasets

In [ ]:
import os
import json
import random
import numpy as np
import pandas as pd
import csv

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

## 2) Load the `dair-ai/emotion` dataset from HuggingFace

The dataset contains 20,000 short English Twitter-style texts labelled with one of six emotions:

| int | emotion  |
|-----|----------|
| 0   | sadness  |
| 1   | joy      |
| 2   | love     |
| 3   | anger    |
| 4   | fear     |
| 5   | surprise |

Pre-split: train (16,000), validation (2,000), test (2,000).


In [ ]:
from datasets import load_dataset

emotion_dataset = load_dataset("dair-ai/emotion")
print(emotion_dataset)

EMOTION_NAMES = {0: "sadness", 1: "joy", 2: "love", 3: "anger", 4: "fear", 5: "surprise"}

def hf_split_to_df(split_data, id_offset=0):
    records = []
    for i, ex in enumerate(split_data):
        records.append({
            "id": id_offset + i,
            "text": str(ex["text"]).strip(),
            "raw_label": ex["label"],
            "emotion": EMOTION_NAMES[ex["label"]],
        })
    return pd.DataFrame(records)

train_raw_df = hf_split_to_df(emotion_dataset["train"], id_offset=0)
val_raw_df   = hf_split_to_df(emotion_dataset["validation"], id_offset=len(emotion_dataset["train"]))
test_raw_df  = hf_split_to_df(emotion_dataset["test"],
                               id_offset=len(emotion_dataset["train"]) + len(emotion_dataset["validation"]))

print("train:", train_raw_df.shape)
print("val:",   val_raw_df.shape)
print("test:",  test_raw_df.shape)
print(train_raw_df["emotion"].value_counts())


## 3) Build the multi-class emotion dataset (6 emotions)


In [ ]:
TARGET_SLUG = "emotion"

def prepare_df(df):
    out = df.copy()
    out["label"] = out["raw_label"]  # use original 6-class labels (0-5)
    out["title"] = out["text"].fillna("").astype(str).str.replace(r"\s+", " ", regex=True).str.strip()
    return out

train_df = prepare_df(train_raw_df)
val_df   = prepare_df(val_raw_df)
test_df  = prepare_df(test_raw_df)

print("Train label distribution:")
print(train_df["label"].value_counts().sort_index())
print(train_df[["id", "emotion", "label", "title"]].head(3))


In [ ]:
def to_lts_format(df):
    out = df.copy()
    out["title"] = out["title"].fillna("").astype(str).str.replace(r"\s+", " ", regex=True).str.strip()
    keep_cols = ["id", "title", "label", "emotion"]
    return out[keep_cols]

train_lts = to_lts_format(train_df)
val_lts   = to_lts_format(val_df)
test_lts  = to_lts_format(test_df)

train_inner_path = os.path.join(DATA_PROCESSED_DIR, f"train_inner_emotions_{TARGET_SLUG}.csv")
val_path         = os.path.join(DATA_PROCESSED_DIR, f"val_emotions_{TARGET_SLUG}.csv")
test_path        = os.path.join(DATA_PROCESSED_DIR, f"test_emotions_{TARGET_SLUG}.csv")

train_lts.to_csv(train_inner_path, index=False, quoting=csv.QUOTE_ALL)
val_lts.to_csv(val_path,           index=False, quoting=csv.QUOTE_ALL)
test_lts.to_csv(test_path,         index=False, quoting=csv.QUOTE_ALL)

print(train_inner_path)
print(val_path)
print(test_path)
print("Train label counts:")
print(train_lts["label"].value_counts())
print(train_lts[["id", "emotion", "label", "title"]].head(3))


## 4) Build the few-shot prompt set for the labeler

This cell selects 2 examples per emotion class (12 total) as curated few-shot demonstrations.


In [ ]:
def _clean_text(text, max_chars=140):
    return str(text).strip().replace("\n", " ")[:max_chars]

def build_few_shot_from_split(df, n_per_class=2,
                               id_col="id", text_col="title", label_col="label"):
    examples = []
    for cls in sorted(df[label_col].unique()):
        cls_rows = df[df[label_col] == cls].head(n_per_class)
        for _, row in cls_rows.iterrows():
            examples.append({
                "id": int(row[id_col]),
                "text": _clean_text(row[text_col], max_chars=140),
                "label": str(int(row[label_col]))
            })
    return examples

few_shot_examples = build_few_shot_from_split(train_lts)

few_shot_path = os.path.join(PROMPTS_DIR, f"few_shot_examples_{TARGET_SLUG}.json")
with open(few_shot_path, "w", encoding="utf-8") as f:
    json.dump(few_shot_examples, f, indent=2, ensure_ascii=False)

print("Saved few-shot examples to:", few_shot_path)
print("Number of examples:", len(few_shot_examples))
few_shot_examples[:3]


## 5) Use the Python source files from the repository

This notebook does **not** rewrite the pipeline modules. Instead, it uses the authoritative source files stored in:

- `emotions_rec/src/`
- `emotions_rec/prompts/`

The next cells verify that those files exist before running the experiment.


In [ ]:
EXPECTED_SRC_FILES = [
    "preprocessing.py",
    "LDA.py",
    "labeling.py",
    "random_sampling.py",
    "thompson_sampling.py",
    "fine_tune.py",
    "tiered_labels.py",
    "main_cluster_binary.py",
    "main_cluster_hierarchical.py",
]

missing = []
for fname in EXPECTED_SRC_FILES:
    fpath = os.path.join(SRC_DIR, fname)
    exists = os.path.exists(fpath)
    print(f"{fname}: {'FOUND' if exists else 'MISSING'}")
    if not exists:
        missing.append(fname)

if missing:
    raise FileNotFoundError(f"Missing source files in src/: {missing}")
else:
    print("\nAll required source files are present.")


In [ ]:
few_shot_repo_path = os.path.join(PROMPTS_DIR, f"few_shot_examples_{TARGET_SLUG}.json")
print("Few-shot prompt file:", few_shot_repo_path)
print("Exists?", os.path.exists(few_shot_repo_path))

if os.path.exists(few_shot_repo_path):
    with open(few_shot_repo_path, "r", encoding="utf-8") as f:
        few_shot_preview = json.load(f)
    print("Number of examples:", len(few_shot_preview))
    print("First two examples:")
    print(few_shot_preview[:2])


## 6) Reset state files before a fresh run

This step is important.

The pipeline stores root-level sampler state files such as:
- `selected_ids.txt`
- `wins.txt`
- `losses.txt`
- `positive_data.csv`

It also writes generated run artifacts to `data/processed/`, such as:
- `*_model_results.json`
- `*_training_data.csv`
- `*_data_labeled.csv`
- `*_lda.csv`

If you rerun without clearing them, sampling may behave strangely or appear to reuse old state.


In [ ]:
for fname in [
    "selected_ids.txt",
    "wins.txt",
    "losses.txt",
    "minority_data.csv",
]:
    path = os.path.join(EXPERIMENT_ROOT, fname)
    if os.path.exists(path):
        os.remove(path)
        print("Removed:", path)

for fname in [
    f"train_inner_emotions_{TARGET_SLUG}_model_results.json",
    f"train_inner_emotions_{TARGET_SLUG}_training_data.csv",
    f"train_inner_emotions_{TARGET_SLUG}_data_labeled.csv",
    f"train_inner_emotions_{TARGET_SLUG}_lda.csv",
]:
    path = os.path.join(DATA_PROCESSED_DIR, fname)
    if os.path.exists(path):
        os.remove(path)
        print("Removed:", path)


## 7) Sanity-check the Python files

This catches syntax problems before the long run starts.


In [ ]:
import subprocess

compile_targets = [os.path.join(SRC_DIR, f) for f in [
    "preprocessing.py",
    "LDA.py",
    "labeling.py",
    "random_sampling.py",
    "thompson_sampling.py",
    "fine_tune.py",
    "tiered_labels.py",
    "main_cluster_binary.py",
    "main_cluster_hierarchical.py",
]]

result = subprocess.run(
    ["python", "-m", "py_compile", *compile_targets],
    capture_output=True,
    text=True,
    cwd=EXPERIMENT_ROOT,
)

if result.returncode == 0:
    print("All files compiled successfully.")
else:
    print("Compilation failed:")
    print(result.stderr)


In [ ]:
%cd "{EXPERIMENT_ROOT}"

## 8) Run the hierarchical experiment

Entry points:

- `tiered_labels.py` — shared label maps (read this first)
- `main_cluster_binary.py` — tier 1 only (pos vs neg)
- `main_cluster_hierarchical.py` — trains tier 1, then tier 2 neg (3 classes), tier 2 pos (joy vs love), or runs `eval`

Option A: one command trains all three heads (`train` subcommand).  
Option B: run `main_cluster_binary.py` alone for tier-1 experiments only.


In [ ]:
# Option A: Train all 3 stages (binary + neg_sub + pos_sub) in one go
!python src/main_cluster_hierarchical.py train \
  -sample_size 300 \
  -filename "data/processed/train_inner_emotions_emotion" \
  -val_path "data/processed/val_emotions_emotion.csv" \
  -sampling "thompson" \
  -filter_label True \
  -model_finetune "bert-base-uncased" \
  -labeling "qwen" \
  -metric "f1_macro" \
  -cluster_size 10 \
  -few_shot_path "prompts/few_shot_examples_emotion.json" \
  -hf_model_id "Qwen/Qwen2.5-7B-Instruct" \
  -max_iterations 8 \
  -confidence_threshold 0.35


## 8b) Evaluate the hierarchical model

After training, update the model paths below with the best checkpoint from each stage's output.


In [ ]:
# Update these paths to your best checkpoints from the training output
!python src/main_cluster_hierarchical.py eval \
  -val_path "data/processed/test_emotions_emotion.csv" \
  -binary_model "models/binary_fine_tunned_0_bandit_0" \
  -neg_model "models/neg_sub_fine_tunned_0_bandit_0" \
  -pos_model "models/pos_sub_fine_tunned_0_bandit_0"


### Option B: Run binary only

If you just want the binary (positive/negative) classifier without the hierarchical cascade:


In [ ]:
# Binary-only run (no sub-classifiers)
!python src/main_cluster_binary.py \
  -sample_size 300 \
  -filename "data/processed/train_inner_emotions_emotion" \
  -val_path "data/processed/val_emotions_emotion.csv" \
  -sampling "thompson" \
  -filter_label True \
  -model_finetune "bert-base-uncased" \
  -labeling "qwen" \
  -metric "f1_macro" \
  -cluster_size 10 \
  -few_shot_path "prompts/few_shot_examples_emotion.json" \
  -hf_model_id "Qwen/Qwen2.5-7B-Instruct" \
  -max_iterations 8 \
  -confidence_threshold 0.35


In [ ]:
import pandas as pd

labeled_path = os.path.join(DATA_PROCESSED_DIR, f"train_inner_emotions_{TARGET_SLUG}_data_labeled.csv")
df = pd.read_csv(labeled_path)
print(df["label"].value_counts(dropna=False))

if "true_label" in df.columns:
    print(df["true_label"].value_counts(dropna=False))
    print("Agreement:", (df["label"] == df["true_label"]).mean())

print(df[["title", "label"]].head(10))


In [ ]:
clustered_path = os.path.join(DATA_PROCESSED_DIR, f"train_inner_emotions_{TARGET_SLUG}_lda.csv")
clustered_df = pd.read_csv(clustered_path)
print(clustered_df.columns.tolist())
print(clustered_df[["id", "emotion", "label", "label_cluster", "title"]].head(10).to_string(index=False))


In [ ]:
EMOTION_NAMES = {0: 'sadness', 1: 'joy', 2: 'love', 3: 'anger', 4: 'fear', 5: 'surprise'}
cluster_summary = (
    clustered_df.groupby("label_cluster")["label"]
    .value_counts()
    .unstack(fill_value=0)
)
cluster_summary.columns = [EMOTION_NAMES.get(c, c) for c in cluster_summary.columns]
cluster_summary["total"] = cluster_summary.sum(axis=1)
print(cluster_summary)


## 9) Summarize final run outputs

Use this cell after the run finishes to inspect the saved evaluation metrics and cluster composition.


In [ ]:
import json
import pandas as pd

results_path = os.path.join(DATA_PROCESSED_DIR, f"train_inner_emotions_{TARGET_SLUG}_model_results.json")
if os.path.exists(results_path):
    with open(results_path, "r", encoding="utf-8") as f:
        results = json.load(f)
    print("Saved model results keys:", list(results.keys()))
    flat = []
    for k, vals in results.items():
        for v in vals:
            row = {"run_key": k}
            row.update(v)
            flat.append(row)
    results_df = pd.DataFrame(flat)
    display(results_df)
else:
    print("No model results json found:", results_path)

labeled_path = os.path.join(DATA_PROCESSED_DIR, f"train_inner_emotions_{TARGET_SLUG}_data_labeled.csv")
if os.path.exists(labeled_path):
    labeled_df = pd.read_csv(labeled_path)
    print("\nPseudo-label distribution:")
    print(labeled_df["label"].value_counts(dropna=False))
    if "true_label" in labeled_df.columns:
        print("\nTrue-label distribution:")
        print(labeled_df["true_label"].value_counts(dropna=False))
        print("Agreement:", (labeled_df["label"] == labeled_df["true_label"]).mean())
else:
    print("No labeled csv found:", labeled_path)
